# 03 — Results & robustness charts

Loads every `outputs/*/eval_*.csv` written by
`python -m deepfake_kyc.evaluate` and draws the clean-test-set results table
plus the robustness curves (accuracy/AUC vs. degradation severity). This
notebook only reads CSVs that evaluation already wrote — it does not run the
model itself.


In [ ]:
from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "deepfake_kyc").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Open this notebook from inside the "
        "project (e.g. from notebooks/) and select the .venv kernel."
    )


PROJECT_ROOT = find_project_root()

import pandas as pd

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
eval_files = sorted(OUTPUTS_DIR.glob("*/eval_*.csv")) if OUTPUTS_DIR.is_dir() else []

print(f"Found {len(eval_files)} eval CSV(s) under {OUTPUTS_DIR}")
for f in eval_files:
    print(" ", f.relative_to(PROJECT_ROOT))


If nothing was found, generate at least one evaluation first:

```bash
python -m deepfake_kyc.train --config configs/experiments/02_effnet_frozen.yaml
python -m deepfake_kyc.evaluate --checkpoint outputs/02_effnet_frozen/best.pt --degradation all
```

Each `evaluate` call writes one CSV (clean plus every requested
degradation/severity) next to the checkpoint it was run on.


In [ ]:
if eval_files:
    frames = []
    for f in eval_files:
        df = pd.read_csv(f)
        df["experiment"] = f.parent.name
        df["source_file"] = f.name
        frames.append(df)
    results = pd.concat(frames, ignore_index=True)
    print(f"Combined {len(results)} rows from {len(eval_files)} file(s).")
else:
    results = pd.DataFrame()
    print("No eval CSVs found yet — run deepfake_kyc.evaluate first (see the cell above).")
results.head()


## Clean test-set results

One row per experiment, sorted by ROC AUC.


In [ ]:
if not results.empty:
    clean = results[results["condition"] == "clean"].sort_values("roc_auc", ascending=False)
    display_cols = ["experiment", "accuracy", "precision", "recall", "f1", "roc_auc", "far", "frr"]
    clean[[c for c in display_cols if c in clean.columns]]
else:
    print("Skipped: no results loaded.")


## Robustness curves

`far` (false acceptance rate) is the share of fakes the model lets through as
real; `frr` (false rejection rate) is the share of real customers wrongly
flagged as fake. Change `metric` below to look at either instead of
`roc_auc`.


In [ ]:
if not results.empty:
    import matplotlib.pyplot as plt

    degraded = results[results["condition"] != "clean"]
    conditions = sorted(degraded["condition"].unique())
    metric = "roc_auc"  # try "accuracy", "far", or "frr"

    fig, axes = plt.subplots(1, len(conditions), figsize=(4 * len(conditions), 3.5), sharey=True)
    if len(conditions) == 1:
        axes = [axes]
    for ax, condition in zip(axes, conditions):
        subset = degraded[degraded["condition"] == condition]
        for experiment, group in subset.groupby("experiment"):
            group = group.sort_values("severity")
            ax.plot(group["severity"], group[metric], marker="o", label=experiment)
        ax.set_title(condition)
        ax.set_xlabel("severity")
    axes[0].set_ylabel(metric)
    axes[-1].legend(fontsize=8, loc="best")
    fig.suptitle(f"{metric} vs. degradation severity")
    fig.tight_layout()
else:
    print("Skipped: no results loaded.")


## Next

[04_error_analysis.ipynb](04_error_analysis.ipynb) looks at individual mistakes
from a trained checkpoint, with Grad-CAM.
